# VLM + SAM 2: text-guided object search

Цель проекта: исследовать пайплайн поиска и сегментации объектов по текстовому запросу. На вход подается изображение или видео и фраза вроде `a bicycle`, на выходе получаются найденные области, маски объектов и метрики качества.

Проект сравнивает несколько подходов к open-vocabulary detection, подбирает prompt templates и пороги детекции, а финальная оценка проводится на COCO val2017. Отдельно считается oracle baseline по COCO boxes, чтобы понять, где теряется качество: на поиске объекта или на сегментации.

## Литература

- MDETR, ICCV 2021: текст используется как условие для детекции, а не как постфильтр над закрытым набором классов. https://openaccess.thecvf.com/content/ICCV2021/html/Kamath_MDETR_-_Modulated_Detection_for_End-to-End_Multi-Modal_Understanding_ICCV_2021_paper.html
- GLIP, CVPR 2022: detection и phrase grounding объединяются в одну задачу grounded language-image pre-training. https://openaccess.thecvf.com/content/CVPR2022/papers/Li_Grounded_Language-Image_Pre-Training_CVPR_2022_paper.pdf
- Segment Anything, ICCV 2023: сегментация формулируется как promptable task, где box prompt является сильным способом получить маску. https://openaccess.thecvf.com/content/ICCV2023/html/Kirillov_Segment_Anything_ICCV_2023_paper.html
- YOLO-World, CVPR 2024: open-vocabulary detection ускоряется через region-text contrastive learning и YOLO-подобную архитектуру. https://arxiv.org/abs/2401.17270
- Grounding DINO: text-conditioned detector, который удобно использовать как grounding-модуль перед SAM 2. https://arxiv.org/abs/2303.05499

Логика эксперимента взята из этих работ: отдельно оценивать text grounding, отдельно promptable segmentation и сравнивать open-vocabulary detectors по качеству и времени.

### Установка

In [ ]:
%pip install -q --no-cache-dir --force-reinstall "Pillow==11.3.0"
%pip install -q -U --no-cache-dir "transformers>=4.57.6" accelerate opencv-python matplotlib pandas requests tqdm pycocotools ultralytics

### Импорты

In [ ]:
import os
import time
import random
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt
import PIL

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, Video
from pycocotools.coco import COCO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import Sam2Processor, Sam2Model

print("Pillow:", PIL.__version__)

### Параметры

In [ ]:
OUT_DIR = Path("outputs")
DATA_DIR = Path("data")
COCO_IMG_DIR = DATA_DIR / "coco_val2017_images"
OUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
COCO_IMG_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

DINO_ID = "IDEA-Research/grounding-dino-tiny"
SAM2_ID = "facebook/sam2.1-hiera-tiny"
YOLO_WORLD_ID = "yolov8s-world.pt"

TEXT_LABELS = ["a cat", "a remote control"]

SEED = 42
BENCH_MODE = "portfolio"  # "debug", "portfolio", "fuller"
MODE_CONFIG = {
    "debug": {"samples_per_category": 1, "max_tasks": 24, "tune_tasks": 8, "test_tasks": 16},
    "portfolio": {"samples_per_category": 2, "max_tasks": 140, "tune_tasks": 32, "test_tasks": 96},
    "fuller": {"samples_per_category": 4, "max_tasks": 320, "tune_tasks": 64, "test_tasks": 220},
}
CFG = MODE_CONFIG[BENCH_MODE]

MIN_OBJECT_AREA = 2500
MAX_OBJECTS = 8
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25
YOLO_CONF = 0.25

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Загрузка моделей

In [ ]:
print("loading Grounding DINO")
dino_processor = AutoProcessor.from_pretrained(DINO_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to(device)
dino_model.eval()

print("loading SAM 2")
sam_processor = Sam2Processor.from_pretrained(SAM2_ID)
sam_model = Sam2Model.from_pretrained(SAM2_ID).to(device)
sam_model.eval()

print("ready")

### YOLO-World

In [ ]:
USE_YOLO_WORLD = True
try:
    from ultralytics import YOLO
    yolo_world = YOLO(YOLO_WORLD_ID)
    print("YOLO-World loaded")
except Exception as e:
    USE_YOLO_WORLD = False
    yolo_world = None
    print("YOLO-World skipped:", repr(e))

### Загрузка изображений

In [ ]:
def load_image_from_url(url):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    return Image.open(response.raw).convert("RGB")

IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image_from_url(IMAGE_URL)
image

## Детекторы

### Grounding DINO

In [ ]:
def detect_grounding_dino(img, labels, box_threshold=0.35, text_threshold=0.25, max_objects=10):
    if isinstance(labels, str):
        labels = [labels]

    inputs = dino_processor(
        images=img,
        text=[labels],
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        outputs = dino_model(**inputs)

    result = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[img.size[::-1]],
    )[0]

    boxes = result["boxes"].detach().cpu().numpy()
    scores = result["scores"].detach().cpu().numpy()
    raw_names = result.get("text_labels", result.get("labels", []))
    names = [str(x) for x in raw_names]

    if len(scores) == 0:
        return boxes, scores, names

    order = np.argsort(-scores)[:max_objects]
    return boxes[order], scores[order], [names[i] for i in order]

### YOLO-World detector

In [ ]:
def clean_class_name(label):
    label = label.strip().lower()
    for prefix in ["a ", "an ", "the "]:
        if label.startswith(prefix):
            label = label[len(prefix):]
    return label

def detect_yolo_world(img, labels, conf=0.25, iou=0.50, max_objects=10):
    if not USE_YOLO_WORLD:
        return np.zeros((0, 4), dtype=float), np.array([]), []

    if isinstance(labels, str):
        labels = [labels]

    class_names = [clean_class_name(x) for x in labels]
    yolo_world.set_classes(class_names)

    arr = np.array(img.convert("RGB"))
    result = yolo_world.predict(arr, conf=conf, iou=iou, verbose=False)[0]

    if result.boxes is None or len(result.boxes) == 0:
        return np.zeros((0, 4), dtype=float), np.array([]), []

    boxes = result.boxes.xyxy.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy()
    cls_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
    names = [result.names.get(int(c), class_names[int(c)] if int(c) < len(class_names) else "object") for c in cls_ids]

    order = np.argsort(-scores)[:max_objects]
    return boxes[order], scores[order], [names[i] for i in order]

### COCO oracle boxes

In [ ]:
def coco_boxes_xyxy(coco, image_id, category_name):
    cat_ids = coco.getCatIds(catNms=[category_name])
    ann_ids = coco.getAnnIds(imgIds=[image_id], catIds=cat_ids, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    boxes = []
    for ann in anns:
        x, y, w, h = ann["bbox"]
        boxes.append([x, y, x + w, y + h])

    return np.array(boxes, dtype=float)

## Сегментация

### SAM 2 masks

In [ ]:
def segment_with_sam2(img, boxes):
    if len(boxes) == 0:
        return np.zeros((0, img.height, img.width), dtype=bool)

    inputs = sam_processor(
        images=img,
        input_boxes=[boxes.tolist()],
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        if device == "cuda":
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = sam_model(**inputs, multimask_output=False)
        else:
            outputs = sam_model(**inputs, multimask_output=False)

    masks = sam_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"],
    )[0]

    if masks.ndim == 4:
        masks = masks[:, 0]
    return masks.numpy() > 0

### Визуализация

In [ ]:
COLORS = [
    (255, 70, 70),
    (80, 170, 255),
    (70, 210, 130),
    (255, 210, 70),
    (190, 120, 255),
    (255, 140, 70),
]

def draw_result(img, boxes, masks, labels, scores, alpha=0.45):
    drawn = np.array(img.convert("RGB")).copy()

    for i, box in enumerate(boxes):
        color = np.array(COLORS[i % len(COLORS)], dtype=np.uint8)

        if i < len(masks):
            mask = masks[i]
            drawn[mask] = (drawn[mask] * (1 - alpha) + color * alpha).astype(np.uint8)

        x1, y1, x2, y2 = box.astype(int)
        cv2.rectangle(drawn, (x1, y1), (x2, y2), color.tolist(), 2)

        text = f"{labels[i]} {scores[i]:.2f}"
        y_text = max(18, y1 - 6)
        cv2.putText(drawn, text, (x1, y_text), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (20, 20, 20), 3, cv2.LINE_AA)
        cv2.putText(drawn, text, (x1, y_text), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

    return Image.fromarray(drawn)

def show_before_after(before, after, title="result"):
    plt.figure(figsize=(13, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(before)
    plt.axis("off")
    plt.title("original")

    plt.subplot(1, 2, 2)
    plt.imshow(after)
    plt.axis("off")
    plt.title(title)
    plt.show()

### Пайплайн

In [ ]:
def run_detector(img, labels, detector="grounding_dino", box_threshold=0.35, text_threshold=0.25, yolo_conf=0.25):
    if detector == "grounding_dino":
        return detect_grounding_dino(
            img,
            labels,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            max_objects=MAX_OBJECTS,
        )

    if detector == "yolo_world":
        return detect_yolo_world(
            img,
            labels,
            conf=yolo_conf,
            max_objects=MAX_OBJECTS,
        )

    raise ValueError(f"unknown detector: {detector}")

def run_text_search_on_image(img, labels=TEXT_LABELS, detector="grounding_dino", **kwargs):
    boxes, scores, names = run_detector(img, labels, detector=detector, **kwargs)
    masks = segment_with_sam2(img, boxes)
    vis = draw_result(img, boxes, masks, names, scores)

    rows = []
    for box, score, name in zip(boxes, scores, names):
        rows.append({
            "detector": detector,
            "label": name,
            "score": float(score),
            "x1": float(box[0]),
            "y1": float(box[1]),
            "x2": float(box[2]),
            "y2": float(box[3]),
        })

    return {
        "boxes": boxes,
        "scores": scores,
        "labels": names,
        "masks": masks,
        "vis": vis,
        "table": pd.DataFrame(rows),
    }

## Демо

### Grounding DINO + SAM 2

In [ ]:
res_dino = run_text_search_on_image(image, TEXT_LABELS, detector="grounding_dino")
show_before_after(image, res_dino["vis"], title="Grounding DINO + SAM 2")
res_dino["table"]

### YOLO-World + SAM 2

In [ ]:
res_yolo = run_text_search_on_image(image, TEXT_LABELS, detector="yolo_world", yolo_conf=YOLO_CONF)
show_before_after(image, res_yolo["vis"], title="YOLO-World + SAM 2")
res_yolo["table"]

### Экспорт демо

In [ ]:
res_dino["vis"].save(OUT_DIR / "demo_grounding_dino_sam2.png")
res_yolo["vis"].save(OUT_DIR / "demo_yolo_world_sam2.png")
print("saved demo images")

### Свое изображение

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# img_path = next(iter(uploaded.keys()))
# my_image = Image.open(img_path).convert("RGB")
# my_labels = ["a person"]
# my_res = run_text_search_on_image(my_image, my_labels, detector="grounding_dino")
# show_before_after(my_image, my_res["vis"], title="my image")
# my_res["table"]

## COCO benchmark

### Аннотации

In [ ]:
def download_file(url, path):
    path = Path(path)
    if path.exists():
        return path

    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    return path

COCO_ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_ZIP = DATA_DIR / "annotations_trainval2017.zip"
COCO_ANN = DATA_DIR / "annotations" / "instances_val2017.json"

if not COCO_ANN.exists():
    download_file(COCO_ANN_URL, COCO_ZIP)
    with zipfile.ZipFile(COCO_ZIP, "r") as z:
        z.extract("annotations/instances_val2017.json", DATA_DIR)

coco = COCO(str(COCO_ANN))

### COCO tasks

In [ ]:
def build_coco_tasks(coco, samples_per_category=2, max_tasks=140, min_area=2500, seed=42):
    rng = random.Random(seed)
    tasks = []

    categories = coco.loadCats(coco.getCatIds())
    categories = sorted(categories, key=lambda x: x["name"])

    for cat in categories:
        cat_id = cat["id"]
        img_ids = coco.getImgIds(catIds=[cat_id])
        candidates = []

        for img_id in img_ids:
            ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=[cat_id], iscrowd=False)
            anns = coco.loadAnns(ann_ids)
            valid_anns = [a for a in anns if a.get("area", 0) >= min_area]
            if not valid_anns:
                continue

            total_area = sum(a.get("area", 0) for a in valid_anns)
            candidates.append({
                "image_id": int(img_id),
                "category": cat["name"],
                "category_id": int(cat_id),
                "instances": len(valid_anns),
                "total_area": float(total_area),
            })

        rng.shuffle(candidates)
        tasks.extend(candidates[:samples_per_category])

    rng.shuffle(tasks)
    tasks = tasks[:max_tasks]
    return pd.DataFrame(tasks)

all_tasks_df = build_coco_tasks(
    coco,
    samples_per_category=CFG["samples_per_category"],
    max_tasks=CFG["max_tasks"],
    min_area=MIN_OBJECT_AREA,
    seed=SEED,
)

all_tasks_df.to_csv(OUT_DIR / "coco_generated_tasks.csv", index=False)
print("tasks:", len(all_tasks_df))
print("unique categories:", all_tasks_df["category"].nunique())
all_tasks_df.head()

### Tune test split

In [ ]:
def make_tune_test_split(tasks_df, tune_tasks=32, test_tasks=96, seed=42):
    df = tasks_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    tune_df = df.iloc[:tune_tasks].copy()
    test_df = df.iloc[tune_tasks:tune_tasks + test_tasks].copy()

    if len(test_df) == 0:
        raise ValueError("test split is empty, increase max_tasks")

    tune_df["split"] = "tune"
    test_df["split"] = "test"
    return tune_df.reset_index(drop=True), test_df.reset_index(drop=True)

tune_tasks_df, test_tasks_df = make_tune_test_split(
    all_tasks_df,
    tune_tasks=CFG["tune_tasks"],
    test_tasks=CFG["test_tasks"],
    seed=SEED,
)

pd.concat([tune_tasks_df, test_tasks_df]).to_csv(OUT_DIR / "coco_tune_test_tasks.csv", index=False)
print("tune tasks:", len(tune_tasks_df), "categories:", tune_tasks_df["category"].nunique())
print("test tasks:", len(test_tasks_df), "categories:", test_tasks_df["category"].nunique())
test_tasks_df.head()

### Ground truth

In [ ]:
IMAGE_CACHE = {}
GT_CACHE = {}

def get_coco_image(coco, image_id):
    image_id = int(image_id)
    if image_id in IMAGE_CACHE:
        return IMAGE_CACHE[image_id]

    info = coco.loadImgs([image_id])[0]
    file_name = info["file_name"]
    local_path = COCO_IMG_DIR / file_name

    if not local_path.exists():
        url = info.get("coco_url") or f"http://images.cocodataset.org/val2017/{file_name}"
        download_file(url, local_path)

    img = Image.open(local_path).convert("RGB")
    IMAGE_CACHE[image_id] = (img, info)
    return img, info

def get_gt_mask(coco, image_id, category_name):
    image_id = int(image_id)
    key = (image_id, category_name)
    if key in GT_CACHE:
        return GT_CACHE[key]

    _, info = get_coco_image(coco, image_id)
    cat_ids = coco.getCatIds(catNms=[category_name])
    ann_ids = coco.getAnnIds(imgIds=[image_id], catIds=cat_ids, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    mask = np.zeros((info["height"], info["width"]), dtype=bool)
    valid_anns = [a for a in anns if a.get("area", 0) >= MIN_OBJECT_AREA]
    for ann in valid_anns:
        mask |= coco.annToMask(ann).astype(bool)

    boxes = coco_boxes_xyxy(coco, image_id, category_name)
    GT_CACHE[key] = (mask, boxes)
    return mask, boxes

### Метрики

In [ ]:
def union_masks(masks):
    if len(masks) == 0:
        return None
    return np.any(masks, axis=0)

def mask_iou(pred_mask, gt_mask):
    if pred_mask is None:
        return 0.0
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return float(intersection / union) if union else 0.0

def pixel_precision_recall(pred_mask, gt_mask):
    if pred_mask is None:
        return 0.0, 0.0

    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()

    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    return precision, recall

def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=float)

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]

    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

def box_recall_at_50(pred_boxes, gt_boxes):
    if len(gt_boxes) == 0:
        return 0.0
    if len(pred_boxes) == 0:
        return 0.0
    ious = box_iou_matrix(gt_boxes, pred_boxes)
    return float((ious.max(axis=1) >= 0.50).mean())

def task_ap50(pred_boxes, pred_scores, gt_boxes):
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return 0.0

    order = np.argsort(-np.asarray(pred_scores))
    pred_boxes = np.asarray(pred_boxes)[order]
    matched = np.zeros(len(gt_boxes), dtype=bool)
    ious = box_iou_matrix(pred_boxes, gt_boxes)

    tp = []
    fp = []
    for pred_i in range(len(pred_boxes)):
        best_gt = int(np.argmax(ious[pred_i])) if len(gt_boxes) else -1
        best_iou = ious[pred_i, best_gt] if best_gt >= 0 else 0.0
        if best_iou >= 0.50 and not matched[best_gt]:
            tp.append(1)
            fp.append(0)
            matched[best_gt] = True
        else:
            tp.append(0)
            fp.append(1)

    tp = np.cumsum(tp)
    fp = np.cumsum(fp)
    recalls = tp / max(len(gt_boxes), 1)
    precisions = tp / np.maximum(tp + fp, 1)

    recall_points = np.linspace(0, 1, 101)
    interp = []
    for r in recall_points:
        valid = precisions[recalls >= r]
        interp.append(valid.max() if len(valid) else 0.0)
    return float(np.mean(interp))

### Оценка подхода

In [ ]:
def make_query(category, template="a {category}"):
    return template.format(category=category)

def sync_cuda():
    if device == "cuda":
        torch.cuda.synchronize()

def iter_tasks(tasks):
    if isinstance(tasks, pd.DataFrame):
        for _, row in tasks.iterrows():
            yield row.to_dict()
    else:
        yield from tasks

def evaluate_approach(tasks, approach, prompt_template="a {category}", box_threshold=0.35, text_threshold=0.25, yolo_conf=0.25, save_preview=False):
    rows = []
    task_list = list(iter_tasks(tasks))

    for item in tqdm(task_list, total=len(task_list)):
        image_id = int(item["image_id"])
        category = item["category"]
        query = make_query(category, prompt_template)

        img, _ = get_coco_image(coco, image_id)
        gt_mask, gt_boxes = get_gt_mask(coco, image_id, category)

        sync_cuda()
        start = time.perf_counter()

        if approach == "oracle_boxes_sam2":
            boxes = gt_boxes
            scores = np.ones(len(boxes), dtype=float)
            names = [category] * len(boxes)
            query_used = "COCO GT boxes"
        elif approach == "grounding_dino_sam2":
            boxes, scores, names = detect_grounding_dino(
                img,
                [query],
                box_threshold=box_threshold,
                text_threshold=text_threshold,
                max_objects=MAX_OBJECTS,
            )
            query_used = query
        elif approach == "yolo_world_sam2":
            boxes, scores, names = detect_yolo_world(
                img,
                [category],
                conf=yolo_conf,
                max_objects=MAX_OBJECTS,
            )
            query_used = category
        else:
            raise ValueError(approach)

        masks = segment_with_sam2(img, boxes)
        sync_cuda()
        elapsed = time.perf_counter() - start

        pred_mask = union_masks(masks)
        miou = mask_iou(pred_mask, gt_mask)
        pix_p, pix_r = pixel_precision_recall(pred_mask, gt_mask)
        b_recall = box_recall_at_50(boxes, gt_boxes)
        ap50 = task_ap50(boxes, scores, gt_boxes)

        preview_path = ""
        if save_preview:
            vis = draw_result(img, boxes, masks, names, scores)
            preview_path = OUT_DIR / f"{approach}_{image_id}_{category.replace(' ', '_')}.png"
            vis.save(preview_path)

        rows.append({
            "approach": approach,
            "split": item.get("split", ""),
            "image_id": image_id,
            "category": category,
            "query": query_used,
            "box_threshold": box_threshold,
            "text_threshold": text_threshold,
            "yolo_conf": yolo_conf,
            "objects_found": len(boxes),
            "gt_objects": len(gt_boxes),
            "best_score": float(np.max(scores)) if len(scores) else 0.0,
            "box_recall_50": b_recall,
            "task_ap50": ap50,
            "mask_iou": miou,
            "pixel_precision": pix_p,
            "pixel_recall": pix_r,
            "hit_iou_50": int(miou >= 0.50),
            "time_sec": elapsed,
            "preview": str(preview_path),
        })

    return pd.DataFrame(rows)

### Summary table

In [ ]:
def summarize_metrics(df):
    return pd.Series({
        "tasks": len(df),
        "categories": df["category"].nunique(),
        "mIoU": df["mask_iou"].mean(),
        "Precision@IoU=0.50": df["hit_iou_50"].mean(),
        "box_recall_50": df["box_recall_50"].mean(),
        "task_mAP50": df["task_ap50"].mean(),
        "pixel_precision": df["pixel_precision"].mean(),
        "pixel_recall": df["pixel_recall"].mean(),
        "avg_found_objects": df["objects_found"].mean(),
        "avg_time_sec": df["time_sec"].mean(),
    })

def summarize_by_approach(df):
    rows = []
    for approach, part in df.groupby("approach"):
        row = summarize_metrics(part).to_dict()
        row["approach"] = approach
        rows.append(row)
    return pd.DataFrame(rows).sort_values("mIoU", ascending=False)

## Эксперимент 1

### Baseline check

In [ ]:
baseline_tasks_df = tune_tasks_df.head(min(12, len(tune_tasks_df))).copy()

baseline_df = evaluate_approach(
    baseline_tasks_df,
    approach="grounding_dino_sam2",
    prompt_template="a {category}",
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
    save_preview=False,
)

baseline_summary = summarize_metrics(baseline_df)
baseline_summary

### График подходов

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(["baseline"], [baseline_summary["mIoU"]])
plt.ylabel("mIoU")
plt.title("Baseline check on tune subset")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.show()

### Интерпретация подходов

In [ ]:
print("Это быстрый baseline check на части tune split.")
print(f"Baseline mIoU={baseline_summary['mIoU']:.3f}")
print("Дальше prompt и thresholds подбираются на tune split, а финальное сравнение считается отдельно на test split.")

## Эксперимент 2

### Prompt templates

In [ ]:
PROMPT_TEMPLATES = [
    "{category}",
    "a {category}",
    "a photo of a {category}",
    "the {category} object",
]

prompt_rows = []
for template in PROMPT_TEMPLATES:
    df = evaluate_approach(
        tune_tasks_df,
        approach="grounding_dino_sam2",
        prompt_template=template,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        save_preview=False,
    )
    s = summarize_metrics(df)
    prompt_rows.append({"prompt_template": template, **s.to_dict()})

prompt_sweep_df = pd.DataFrame(prompt_rows).sort_values(["mIoU", "box_recall_50"], ascending=False)
prompt_sweep_df.to_csv(OUT_DIR / "prompt_template_sweep.csv", index=False)
prompt_sweep_df

### Prompt result

In [ ]:
best_prompt_template = prompt_sweep_df.iloc[0]["prompt_template"]
print("Best prompt template:", best_prompt_template)
print(f"mIoU={prompt_sweep_df.iloc[0]['mIoU']:.3f}")

## Эксперимент 3

### Threshold sweep

In [ ]:
SWEEP_BOX_THRESHOLDS = [0.25, 0.35, 0.45]
SWEEP_TEXT_THRESHOLDS = [0.20, 0.25, 0.30]

threshold_rows = []
for box_t in SWEEP_BOX_THRESHOLDS:
    for text_t in SWEEP_TEXT_THRESHOLDS:
        df = evaluate_approach(
            tune_tasks_df,
            approach="grounding_dino_sam2",
            prompt_template=best_prompt_template,
            box_threshold=box_t,
            text_threshold=text_t,
            save_preview=False,
        )
        s = summarize_metrics(df)
        threshold_rows.append({
            "box_threshold": box_t,
            "text_threshold": text_t,
            **s.to_dict(),
        })

threshold_sweep_df = pd.DataFrame(threshold_rows).sort_values(["mIoU", "Precision@IoU=0.50"], ascending=False)
threshold_sweep_df.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)
threshold_sweep_df

### Sweep plot

In [ ]:
plt.figure(figsize=(8, 4))
for text_t, part in threshold_sweep_df.sort_values("box_threshold").groupby("text_threshold"):
    plt.plot(part["box_threshold"], part["mIoU"], marker="o", label=f"text={text_t}")

plt.xlabel("box_threshold")
plt.ylabel("mIoU")
plt.title("Grounding DINO threshold sweep")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Лучшая конфигурация

In [ ]:
best_cfg = threshold_sweep_df.iloc[0]
BASELINE_PROMPT = "a {category}"
BEST_BOX_THRESHOLD = float(best_cfg["box_threshold"])
BEST_TEXT_THRESHOLD = float(best_cfg["text_threshold"])

print(f"Baseline: prompt='{BASELINE_PROMPT}', box={BOX_THRESHOLD}, text={TEXT_THRESHOLD}")
print(f"Baseline tune mIoU={baseline_summary['mIoU']:.3f}, Precision@0.50={baseline_summary['Precision@IoU=0.50']:.3f}")
print(f"Best tuned: prompt='{best_prompt_template}', box={BEST_BOX_THRESHOLD}, text={BEST_TEXT_THRESHOLD}")
print(f"Best tuned tune mIoU={best_cfg['mIoU']:.3f}, Precision@0.50={best_cfg['Precision@IoU=0.50']:.3f}")

## Эксперимент 4

### Финальное сравнение

In [ ]:
approaches = ["oracle_boxes_sam2", "grounding_dino_sam2"]
if USE_YOLO_WORLD:
    approaches.append("yolo_world_sam2")

final_results = []
for approach in approaches:
    df = evaluate_approach(
        test_tasks_df,
        approach=approach,
        prompt_template=best_prompt_template,
        box_threshold=BEST_BOX_THRESHOLD,
        text_threshold=BEST_TEXT_THRESHOLD,
        yolo_conf=YOLO_CONF,
        save_preview=False,
    )
    final_results.append(df)

final_df = pd.concat(final_results, ignore_index=True)
final_summary = summarize_by_approach(final_df)
final_df.to_csv(OUT_DIR / "coco_test_results.csv", index=False)
final_summary.to_csv(OUT_DIR / "coco_test_summary.csv", index=False)
final_summary

### Test plot

In [ ]:
plot_df = final_summary.sort_values("mIoU")
plt.figure(figsize=(8, 4))
plt.barh(plot_df["approach"], plot_df["mIoU"])
plt.xlabel("mIoU")
plt.title("Final COCO val2017 test comparison")
plt.grid(axis="x", alpha=0.3)
plt.show()

### Test interpretation

In [ ]:
oracle_row = final_summary[final_summary["approach"] == "oracle_boxes_sam2"].iloc[0]
dino_row = final_summary[final_summary["approach"] == "grounding_dino_sam2"].iloc[0]
gap_to_oracle = float(oracle_row["mIoU"] - dino_row["mIoU"])

print(f"Grounding DINO + SAM 2: mIoU={dino_row['mIoU']:.3f}, box_recall_50={dino_row['box_recall_50']:.3f}, task_mAP50={dino_row['task_mAP50']:.3f}")
print(f"Oracle boxes + SAM 2: mIoU={oracle_row['mIoU']:.3f}")
print(f"Gap to oracle: {gap_to_oracle:.3f}")

if gap_to_oracle > 0.15:
    print("Основная потеря качества находится в text grounding: SAM 2 получает неполные или неточные boxes.")
else:
    print("Разрыв с oracle небольшой: выбранный detector дает достаточно хорошие boxes, дальше важнее качество масок.")

if USE_YOLO_WORLD and "yolo_world_sam2" in set(final_summary["approach"]):
    yolo_row = final_summary[final_summary["approach"] == "yolo_world_sam2"].iloc[0]
    print(f"YOLO-World + SAM 2: mIoU={yolo_row['mIoU']:.3f}, avg_time={yolo_row['avg_time_sec']:.2f}s")

print("Prompt и thresholds подбирались на tune split. Эта таблица считается на отдельном test split.")

## Разбор ошибок

### Лучшие и худшие случаи

In [ ]:
analysis_df = final_df[final_df["approach"] == "grounding_dino_sam2"].copy()
analysis_df["quality"] = pd.cut(
    analysis_df["mask_iou"],
    bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
    labels=["bad", "weak", "good", "strong"],
)

cols = ["image_id", "category", "query", "objects_found", "gt_objects", "box_recall_50", "task_ap50", "mask_iou", "pixel_precision", "pixel_recall", "quality"]
print("Worst examples")
display(analysis_df.sort_values("mask_iou").head(10)[cols])

print("Best examples")
display(analysis_df.sort_values("mask_iou", ascending=False).head(10)[cols])

### Вывод по ошибкам

In [ ]:
low_box = analysis_df[analysis_df["box_recall_50"] < 0.5]
low_mask = analysis_df[(analysis_df["box_recall_50"] >= 0.5) & (analysis_df["mask_iou"] < 0.5)]

print("Detector problem cases:", len(low_box))
print("Segmentation problem after acceptable box:", len(low_mask))

if len(low_box) > len(low_mask):
    print("На test split чаще проседает text grounding/detection.")
elif len(low_mask) > len(low_box):
    print("На test split чаще проседает сегментация после найденного box.")
else:
    print("Ошибки распределены примерно поровну между детекцией и сегментацией.")

category_report = analysis_df.groupby("category", as_index=False).agg(
    tasks=("image_id", "count"),
    mIoU=("mask_iou", "mean"),
    box_recall_50=("box_recall_50", "mean"),
    task_mAP50=("task_ap50", "mean"),
).sort_values("mIoU")
category_report.to_csv(OUT_DIR / "category_error_report.csv", index=False)
category_report.head(15)

## Видео

### Загрузка видео

In [ ]:
VIDEO_URL = "https://github.com/opencv/opencv/raw/4.x/samples/data/vtest.avi"
VIDEO_PATH = download_file(VIDEO_URL, "vtest.avi")
VIDEO_LABELS = ["a person"]
VIDEO_PATH

### Свое видео

In [ ]:
# from google.colab import files
# uploaded_video = files.upload()
# VIDEO_PATH = next(iter(uploaded_video.keys()))
# VIDEO_LABELS = ["a car"]

### Чтение кадров

In [ ]:
def read_video_frames(video_path, every_n=8, max_frames=45, resize_to=720):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if fps and fps > 1 else 24

    frames = []
    idx = 0
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break

        if idx % every_n == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            if resize_to and w > resize_to:
                scale = resize_to / w
                frame = cv2.resize(frame, (resize_to, int(h * scale)))
            frames.append(Image.fromarray(frame))
        idx += 1

    cap.release()
    return frames, max(3, fps / every_n)

video_frames, out_fps = read_video_frames(VIDEO_PATH, every_n=8, max_frames=45)
print("frames:", len(video_frames), "fps:", round(out_fps, 2))
video_frames[0]

### Обработка видео

In [ ]:
def save_video(frames_rgb, path, fps=6):
    path = str(path)
    first = np.array(frames_rgb[0])
    h, w = first.shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    for frame in frames_rgb:
        arr = np.array(frame)
        writer.write(cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
    writer.release()

def process_video_simple(frames, labels, detector="grounding_dino"):
    out_frames = []
    all_rows = []

    for frame_i, frame in enumerate(tqdm(frames)):
        result = run_text_search_on_image(frame, labels, detector=detector)
        out_frames.append(result["vis"])

        if len(result["table"]):
            tmp = result["table"].copy()
            tmp.insert(0, "frame", frame_i)
            all_rows.append(tmp)

        if device == "cuda" and frame_i % 10 == 0:
            torch.cuda.empty_cache()

    table = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    return out_frames, table

video_out_frames, video_table = process_video_simple(video_frames, VIDEO_LABELS, detector="grounding_dino")
video_table.head()

### Экспорт видео

In [ ]:
video_out_path = OUT_DIR / "video_result.mp4"
save_video(video_out_frames, video_out_path, fps=out_fps)
video_table.to_csv(OUT_DIR / "video_detections.csv", index=False)

print("saved:", video_out_path)
print("saved:", OUT_DIR / "video_detections.csv")
Video(str(video_out_path), embed=True)

## Итог

В финальной версии проект оценивается не на нескольких картинках, а на COCO val2017 tasks. Датасет собирается автоматически из аннотаций: выбираются категории, изображения с достаточно крупными объектами, затем формируются tune и test splits.

Сравнение включает `oracle_boxes_sam2`, `grounding_dino_sam2` и, если загрузился пакет Ultralytics, `yolo_world_sam2`. Такой дизайн показывает не только визуализацию, но и исследовательский вывод: где находится bottleneck пайплайна, как prompt и thresholds влияют на качество, и какой подход лучше на отложенном test split.